# 第五章：Human-in-the-Loop（人工干预）

## 学习目标
- 理解为什么 Agent 需要人工干预
- 用 `interrupt_before` 在节点执行前暂停
- 用 `get_state()` 检查 Agent 当前状态和待执行的操作
- 恢复执行（批准）或拒绝操作
- 用 `interrupt()` 函数实现更灵活的中断逻辑
- 用 `Command(resume=...)` 传递人类反馈

## 0. 环境准备

In [ ]:
from importlib.metadata import version
print(f"langgraph 版本: {version('langgraph')}")

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv("../.env")

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="qwen-plus",
    openai_api_base=os.environ["Qwen_API_BASE"],
    openai_api_key=os.environ["Qwen_API_KEY"],
)

## 1. 为什么需要 Human-in-the-Loop？

上一章的 Agent 完全自主运行——模型决定调用什么工具，代码立刻执行。这在练习时没问题，但在生产环境中很危险：

- 模型可能误判意图，发送不该发的邮件
- 模型可能调用错误的 API，删除重要数据
- 模型可能在高成本操作上做出错误决策

**Human-in-the-Loop（HITL）** 就是在关键步骤暂停执行，让人类审查后再继续。三种控制级别：

| 级别 | 说明 | 适用场景 |
|------|------|----------|
| 完全自主 | Agent 自行决策并执行 | 低风险操作（查询天气、计算） |
| 执行前审批 | Agent 决策，人类批准后才执行 | 中风险操作（发邮件、修改数据） |
| 人类提供输入 | Agent 暂停，等待人类提供信息再继续 | 需要人类判断的环节 |

LangGraph 通过 **Checkpointer + 中断机制** 实现 HITL。Checkpointer 保存图的执行状态，中断让图暂停在某个节点，等待人类操作后恢复。

## 2. interrupt_before：最简单的中断方式

先定义一些工具，包括一个「危险」的 `send_email` 工具。

In [ ]:
from langchain_core.tools import tool

@tool
def multiply(a: int, b: int) -> int:
    """两个整数相乘"""
    return a * b

@tool
def get_weather(city: str) -> str:
    """查询指定城市的天气"""
    weather_data = {
        "北京": "晴天，15°C",
        "上海": "多云，20°C",
        "深圳": "小雨，25°C",
    }
    return weather_data.get(city, f"{city}：暂无数据")

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """发送邮件给指定收件人"""
    # 模拟发送，实际项目中会调用邮件 API
    return f"邮件已发送给 {to}，主题：{subject}"

tools = [multiply, get_weather, send_email]
print(f"已定义 {len(tools)} 个工具: {[t.name for t in tools]}")

### 2.1 对比：没有中断 vs 有中断

先看**没有中断**的 Agent——和第四章一样，工具直接执行。

In [ ]:
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage

# 没有中断的 Agent：工具直接执行
agent_no_hitl = create_react_agent(llm, tools)

result = agent_no_hitl.invoke({
    "messages": [HumanMessage(content="给 test@example.com 发一封邮件，主题是'会议通知'，内容是'明天下午3点开会'")],
})

for msg in result["messages"]:
    print(f"{msg.__class__.__name__}: ", end="")
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"调用 {tc['name']}({tc['args']})")
    else:
        print(msg.content[:200])

邮件直接发出去了！没有任何确认。这在生产环境中很危险。

现在加上中断——在工具执行**前**暂停。

### 2.2 MemorySaver：中断的前提

要让图暂停后还能恢复，必须保存执行状态。`MemorySaver` 是一个内存中的状态存储器（Checkpointer），它记录图每一步的状态。

> 这里只用 `MemorySaver` 实现 HITL。第七章会深入讲解持久化和其他 Checkpointer。

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

checkpointer = MemorySaver()

# 编译时加上 checkpointer 和 interrupt_before
agent = create_react_agent(
    llm,
    tools,
    checkpointer=checkpointer,
    interrupt_before=["tools"],  # 在 tools 节点执行前暂停
)

from IPython.display import Image
Image(agent.get_graph().draw_mermaid_png())

### 2.3 运行并观察中断

In [ ]:
# 每次对话需要一个 thread_id，标识这次对话
config = {"configurable": {"thread_id": "hitl-demo-1"}}

# 发送请求
result = agent.invoke(
    {"messages": [HumanMessage(content="给 test@example.com 发一封邮件，主题是'会议通知'，内容是'明天下午3点开会'")]},
    config,
)

# 查看返回的消息
for msg in result["messages"]:
    print(f"{msg.__class__.__name__}: ", end="")
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"想调用 {tc['name']}({tc['args']})")
    else:
        print(msg.content[:200])

注意：Agent 返回了，但 `send_email` **没有执行**。模型决定了要调用工具，但图在 `tools` 节点执行前暂停了。

### 2.4 用 get_state() 检查 Agent 的决策

In [ ]:
# 检查当前状态
state = agent.get_state(config)

print(f"下一步要执行的节点: {state.next}")
print()

# 查看模型想调用什么工具
last_message = state.values["messages"][-1]
print("模型的决策:")
for tc in last_message.tool_calls:
    print(f"  工具: {tc['name']}")
    print(f"  参数: {tc['args']}")

`state.next` 是 `('tools',)`，说明图暂停在 `tools` 节点之前。我们可以看到模型想发的邮件内容，然后决定是否批准。

### 2.5 批准操作：恢复执行

In [ ]:
# 批准：传入 None 继续执行
result = agent.invoke(None, config)

# 查看完整流程
for msg in result["messages"]:
    print(f"{msg.__class__.__name__}: ", end="")
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"调用 {tc['name']}({tc['args']})")
    else:
        print(msg.content[:200])

### 刚才发生了什么？

完整流程：
1. `invoke(用户消息)` → 模型决定调用 `send_email` → **暂停**（`interrupt_before=["tools"]`）
2. 我们用 `get_state()` 检查模型想做什么
3. 决定批准 → `invoke(None, config)` → 执行工具 → 模型生成最终回答

关键点：
- `invoke(None, config)` 中的 `None` 表示「不添加新输入，继续执行」
- 必须传同一个 `config`（相同的 `thread_id`），才能恢复之前的对话
- Checkpointer 保存了图在中断时的完整状态

## 3. 检查与拒绝操作

批准很简单，但更重要的是：如果我们不同意 Agent 的操作，怎么办？

In [ ]:
# 新的对话线程
config2 = {"configurable": {"thread_id": "hitl-demo-2"}}

# Agent 想发邮件
result = agent.invoke(
    {"messages": [HumanMessage(content="给 boss@company.com 发封邮件，主题是'辞职信'，内容是'我不干了'")]},
    config2,
)

# 检查 Agent 的决策
state = agent.get_state(config2)
last_msg = state.values["messages"][-1]
print(f"Agent 想调用: {last_msg.tool_calls[0]['name']}")
print(f"参数: {last_msg.tool_calls[0]['args']}")
print(f"\n这操作太危险了！我们要拒绝。")

### 3.1 拒绝方式：用 update_state 修改状态

拒绝操作的思路：手动往消息列表里添加一条 `ToolMessage`，告诉模型「工具调用被拒绝」，然后恢复执行。模型收到拒绝信息后会重新生成回答。

In [ ]:
from langchain_core.messages import ToolMessage

# 获取 Agent 想调用的工具信息
last_msg = state.values["messages"][-1]
tool_call_id = last_msg.tool_calls[0]["id"]

# 构造一条拒绝消息，假装工具返回了拒绝结果
rejection_message = ToolMessage(
    content="操作被用户拒绝。请不要发送这封邮件，告诉用户操作已取消。",
    tool_call_id=tool_call_id,
)

# 用 update_state 把拒绝消息注入状态，并标记为 tools 节点的输出
agent.update_state(config2, {"messages": [rejection_message]}, as_node="tools")

print("已注入拒绝消息，tools 节点被我们手动'执行'了")

In [ ]:
# 恢复执行——模型会收到拒绝消息并重新回答
result = agent.invoke(None, config2)

print("Agent 的最终回复:")
print(result["messages"][-1].content)

### 刚才发生了什么？

拒绝流程：
1. Agent 想调用 `send_email` → 暂停
2. 我们检查后决定拒绝
3. 用 `update_state()` 注入一条 `ToolMessage`，内容是「操作被拒绝」
4. `as_node="tools"` 告诉图：这条消息是 `tools` 节点的输出（相当于我们替代了 tools 节点）
5. `invoke(None, config)` 恢复执行 → 模型收到拒绝消息 → 生成新回答

`update_state` 的关键参数：

| 参数 | 说明 |
|------|------|
| `config` | 哪个对话线程 |
| `values` | 要更新的状态值（这里是新消息） |
| `as_node` | 以哪个节点的身份写入（影响图的执行流程） |

## 4. interrupt() 函数：更灵活的中断

`interrupt_before` 只能在节点执行前暂停，粒度是**节点级别**。如果需要在节点**内部**暂停、向人类提问、获取人类输入，就需要 `interrupt()` 函数。

下面构建一个订单处理流水线：
1. 解析订单
2. **人工确认**订单详情
3. 根据确认结果处理订单

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt, Command

# 定义状态
class OrderState(TypedDict):
    order: str           # 原始订单描述
    parsed: dict         # 解析后的订单信息
    approved: bool       # 是否批准
    result: str          # 处理结果

In [ ]:
# 节点 1：解析订单
def parse_order(state: OrderState) -> dict:
    order_text = state["order"]
    # 模拟解析（实际项目中可能用 LLM 解析）
    parsed = {
        "item": "MacBook Pro 16寸",
        "quantity": 1,
        "price": 19999,
        "shipping": "北京市朝阳区",
    }
    print(f"[parse_order] 解析完成: {parsed}")
    return {"parsed": parsed}

# 节点 2：人工确认（使用 interrupt）
def human_confirm(state: OrderState) -> dict:
    parsed = state["parsed"]
    
    # interrupt() 暂停执行，把订单详情发给调用者
    # 当恢复时，decision 就是人类通过 Command(resume=...) 传入的值
    decision = interrupt({
        "question": "请确认以下订单：",
        "item": parsed["item"],
        "quantity": parsed["quantity"],
        "price": f"¥{parsed['price']}",
        "shipping": parsed["shipping"],
        "instruction": "回复 'yes' 确认，或 'no' 取消",
    })
    
    print(f"[human_confirm] 收到人类决策: {decision}")
    return {"approved": decision == "yes"}

# 节点 3：处理订单
def process_order(state: OrderState) -> dict:
    if state["approved"]:
        result = f"订单已确认！{state['parsed']['item']} x{state['parsed']['quantity']}，即将发往 {state['parsed']['shipping']}"
    else:
        result = "订单已取消。"
    print(f"[process_order] {result}")
    return {"result": result}

In [ ]:
# 构建图
builder = StateGraph(OrderState)
builder.add_node("parse_order", parse_order)
builder.add_node("human_confirm", human_confirm)
builder.add_node("process_order", process_order)

builder.add_edge(START, "parse_order")
builder.add_edge("parse_order", "human_confirm")
builder.add_edge("human_confirm", "process_order")
builder.add_edge("process_order", END)

# 注意：interrupt() 需要 checkpointer，但不需要 interrupt_before
checkpointer2 = MemorySaver()
order_graph = builder.compile(checkpointer=checkpointer2)

from IPython.display import Image
Image(order_graph.get_graph().draw_mermaid_png())

### 4.1 运行并观察 interrupt

In [ ]:
config3 = {"configurable": {"thread_id": "order-1"}}

# 第一次调用：会在 human_confirm 节点暂停
result = order_graph.invoke({"order": "我要买一台 MacBook Pro"}, config3)

print(f"\n返回结果: {result}")

In [ ]:
# 检查状态
state = order_graph.get_state(config3)
print(f"下一步: {state.next}")
print(f"当前状态: {state.values}")

# 查看 interrupt 发送的信息
print(f"\n中断信息:")
for task in state.tasks:
    if task.interrupts:
        for intr in task.interrupts:
            print(f"  {intr.value}")

`state.tasks` 里有 `interrupts` 信息，包含我们在 `interrupt()` 中传入的订单详情。这就是 `interrupt()` 传递给调用者的数据。

### 4.2 用 Command(resume=...) 恢复

In [ ]:
# 批准订单：用 Command(resume="yes") 恢复
result = order_graph.invoke(Command(resume="yes"), config3)

print(f"\n最终结果: {result['result']}")

### 4.3 拒绝订单的流程

In [ ]:
# 新线程：测试拒绝
config4 = {"configurable": {"thread_id": "order-2"}}

# 运行到 interrupt 暂停
order_graph.invoke({"order": "我要买一台 MacBook Pro"}, config4)

# 拒绝：resume="no"
result = order_graph.invoke(Command(resume="no"), config4)

print(f"最终结果: {result['result']}")

### 刚才发生了什么？

`interrupt()` 函数的工作方式：

```python
def human_confirm(state):
    decision = interrupt({"question": "确认订单？"})  # <-- 暂停，发送信息给调用者
    # 恢复后，decision = Command(resume=...) 传入的值
    return {"approved": decision == "yes"}
```

1. 图执行到 `interrupt()` 时暂停
2. `interrupt()` 的参数被发送给调用者（通过 `state.tasks[].interrupts`）
3. 调用者用 `Command(resume=value)` 恢复
4. `interrupt()` 返回 `value`，节点继续执行

这比 `interrupt_before` 灵活得多——可以在节点内部的任意位置暂停，还能双向传递数据。

## 5. 实战模式：选择性工具审批

实际项目中，不是所有工具都需要审批。查天气、做计算是安全的，直接执行即可；发邮件是危险的，需要人类确认。

下面构建一个**选择性审批**的 Agent：
- 安全工具（`multiply`、`get_weather`）→ 直接执行
- 危险工具（`send_email`）→ 暂停等待审批

In [ ]:
from typing import Literal
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.prebuilt import ToolNode

# 定义安全和危险工具
safe_tools = ["multiply", "get_weather"]
dangerous_tools = ["send_email"]

llm_with_tools = llm.bind_tools(tools)

# 节点：调用模型
def call_model(state: MessagesState) -> dict:
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

# 路由：根据工具类型决定走向
def route_tools(state: MessagesState) -> Literal["safe_tools", "dangerous_tools", "__end__"]:
    last_message = state["messages"][-1]
    if not last_message.tool_calls:
        return "__end__"
    
    # 检查是否有危险工具调用
    for tc in last_message.tool_calls:
        if tc["name"] in dangerous_tools:
            return "dangerous_tools"  # 只要有一个危险工具，就走审批
    
    return "safe_tools"  # 全部是安全工具

In [ ]:
# 构建图
builder = StateGraph(MessagesState)
builder.add_node("model", call_model)
builder.add_node("safe_tools", ToolNode(tools))        # 安全工具节点
builder.add_node("dangerous_tools", ToolNode(tools))   # 危险工具节点（同一个 ToolNode）

builder.add_edge(START, "model")
builder.add_conditional_edges("model", route_tools)
builder.add_edge("safe_tools", "model")
builder.add_edge("dangerous_tools", "model")

checkpointer3 = MemorySaver()
selective_agent = builder.compile(
    checkpointer=checkpointer3,
    interrupt_before=["dangerous_tools"],  # 只在危险工具节点前中断
)

from IPython.display import Image
Image(selective_agent.get_graph().draw_mermaid_png())

### 5.1 测试安全工具（不中断）

In [ ]:
config5 = {"configurable": {"thread_id": "selective-1"}}

# 安全工具：直接执行，不暂停
result = selective_agent.invoke(
    {"messages": [HumanMessage(content="北京天气怎么样？")]},
    config5,
)

print("安全工具直接执行:")
print(result["messages"][-1].content)

### 5.2 测试危险工具（中断审批）

In [ ]:
config6 = {"configurable": {"thread_id": "selective-2"}}

# 危险工具：会暂停
result = selective_agent.invoke(
    {"messages": [HumanMessage(content="给 test@example.com 发邮件，主题'测试'，内容'hello'")]},
    config6,
)

# 检查状态
state = selective_agent.get_state(config6)
print(f"暂停在: {state.next}")

last_msg = state.values["messages"][-1]
print(f"待审批操作: {last_msg.tool_calls[0]['name']}({last_msg.tool_calls[0]['args']})")

In [ ]:
# 批准
result = selective_agent.invoke(None, config6)

print("审批通过后的结果:")
print(result["messages"][-1].content)

### 刚才发生了什么？

关键设计：
- 把工具分成两个节点：`safe_tools` 和 `dangerous_tools`（它们用同一个 `ToolNode`，只是图中的位置不同）
- 路由函数 `route_tools` 检查工具名，决定走哪条路
- `interrupt_before=["dangerous_tools"]` 只拦截危险路径

```
START → model ──(安全工具)──→ safe_tools ──→ model → ...
                │
                ├──(危险工具)──→ [中断] → dangerous_tools ──→ model → ...
                │
                └──(无工具)──→ END
```

这是生产环境中最常用的 HITL 模式。

## 6. 对比总结

| | `interrupt_before` | `interrupt()` |
|--|-------------------|---------------|
| **粒度** | 节点级别 | 代码行级别 |
| **配置方式** | 编译时指定 `interrupt_before=["node"]` | 在节点函数内调用 `interrupt()` |
| **使用场景** | 标准工具审批 | 自定义交互逻辑（表单确认、多轮对话） |
| **恢复方式** | `graph.invoke(None, config)` | `graph.invoke(Command(resume=value), config)` |
| **数据传递** | 通过 `get_state()` 查看状态 | `interrupt(data)` 主动发送，`resume` 接收 |
| **灵活性** | 一般（只能在节点边界暂停） | 高（节点内任意位置暂停，双向传数据） |

**选择建议**：
- 需要审批工具调用 → `interrupt_before`（简单直接）
- 需要人类提供输入、确认表单、多步交互 → `interrupt()`（灵活强大）

## 常见问题

**Q: 忘记传 `checkpointer` 会怎样？**

中断需要 Checkpointer 来保存状态。如果不传，`interrupt_before` 会被忽略（图不会暂停），`interrupt()` 会报错。

**Q: `thread_id` 写错了 / 用了不同的 `thread_id` 恢复？**

会找不到之前的状态。每个 `thread_id` 是一个独立的对话线程，恢复时必须用同一个。

**Q: 可以多次 `interrupt()` 吗？**

可以。一个节点里可以调用多次 `interrupt()`，每次暂停都需要一次 `Command(resume=...)` 来恢复。

**Q: `interrupt_before` 和 `interrupt_after` 的区别？**

`interrupt_before=["node"]` 在节点执行**前**暂停，`interrupt_after=["node"]` 在节点执行**后**暂停。后者适用于「执行完让人类检查结果」的场景。

## 总结

本章学习了：
- Agent 在生产环境中需要人工干预的原因
- `MemorySaver` 作为 Checkpointer 保存图的执行状态
- `interrupt_before` 在节点执行前暂停，`invoke(None, config)` 恢复
- `get_state()` 检查 Agent 的当前状态和待执行操作
- `update_state()` 手动修改状态来拒绝操作
- `interrupt()` 函数在节点内部灵活中断，双向传递数据
- `Command(resume=...)` 传递人类反馈给 `interrupt()`
- 选择性审批模式：只拦截危险工具，安全工具直接放行

## 下一章预告

**第六章：多 Agent 协作（子图、Supervisor 模式）** —— 单个 Agent 能力有限。下一章学习如何让多个专业 Agent 协同工作，构建更复杂的应用。